# 14 — Integrate canonical archived CopyKAT calls

This notebook integrates the **canonical archived CopyKAT cell calls used for the paper** into the current full-gene H5AD.

## Reproducibility policy

CopyKAT is version- and baseline-sensitive. A later sensitivity rerun with CopyKAT 1.2.5 and known-normal anchors produced a large, dataset-dependent shift in aneuploid calls, including stronger increases in reference/non-malignant-like compartments than in tumor-like compartments. That rerun is therefore retained as a sensitivity analysis rather than replacing the paper-defining result.

The main pipeline is deterministic:

1. Notebook 13 prepares the CopyKAT microbatch inputs and records the historical design.
2. This notebook reads the **archived canonical historical cell-call table** from `metadata/canonical/`.
3. The canonical archive is checksum-validated.
4. Historical hard checkpoints are validated.
5. The canonical calls are transferred to the current H5AD by stable biological cell identity.
6. Downstream Notebook 15 consumes the standard H5AD produced here.

The archived 176-microbatch prediction bundle is retained in `metadata/canonical/` for provenance and audit. The main integration uses the archived final cell-call table so that reproduction does not depend on rerunning a version-sensitive CopyKAT runtime.

## Important

- This notebook does **not** run CopyKAT.
- It does **not** use the later known-normal sensitivity rerun as the primary result.
- It does **not** load the full expression matrix into RAM.
- Existing non-canonical Notebook-14 outputs must be archived with the provided sensitivity archival helper before this notebook is run.

## Two historical CopyKAT-missing cells

The archived final-call table contains exactly two cells with `missing` CopyKAT calls and missing dataset/sample metadata. The current source H5AD retains their biological metadata. For identity matching only, those two raw cell IDs are forced to use the same `UNASSIGNED_METADATA||<raw_cell_id>` stable key on both sides. No biological metadata in the H5AD is erased or modified.


## Historical summary metadata policy

The canonical historical final-cell-call table is the source of metadata used to recreate the historical Notebook-14 summary tables. This matters for the two `missing` CopyKAT cells: their archived Notebook-14 rows contain missing cluster/annotation/dataset/condition/patient metadata, which creates the historical literal `nan` summary groups. The current source H5AD retains biological metadata for those cells, but that metadata is not substituted into the historical summary tables. Conversely, the H5AD itself keeps its current biological metadata unchanged; only CopyKAT columns are added to `/obs`.


## Historical summary reconstruction

Historical Notebook 14 generated its summary tables from the standardized current-analysis metadata attached to target predictions. The two cells with no CopyKAT target prediction entered the final all-cell table with missing summary metadata. To reproduce that behavior exactly, summary metadata are taken from the current canonical H5AD for all cells, then set to missing only for the two canonical CopyKAT-missing cells. This avoids reintroducing older patient-ID aliases present in the archived CopyKAT table while preserving the historical `nan` summary groups. The H5AD biological metadata itself remains unchanged; this adjustment applies only to the temporary summary table.


In [1]:
# =========================
# IMPORTS + PATHS
# =========================
from pathlib import Path
import os
import re
import hashlib
import shutil
import numpy as np
import pandas as pd
import scanpy as sc
import h5py

try:
    from anndata.io import write_elem
except Exception:
    from anndata._io.specs import write_elem


def find_repo(start=None):
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()

    for p in (start, *start.parents):
        if (
            (p / "README.md").exists()
            and (p / "metadata").exists()
            and (p / "data").exists()
        ):
            return p

    return start


PROJECT_DIR = Path(
    os.environ.get(
        "GLIOMA_PROJECT_ROOT",
        find_repo(),
    )
).expanduser().resolve()

METADATA_DIR = PROJECT_DIR / "metadata"
CANONICAL_DIR = METADATA_DIR / "canonical"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed" / "integrated"

SOURCE_H5AD_CANDIDATES = [
    PROCESSED_DIR
    / "GBM_core_4datasets_fullgene_marker_based_annotation_bias_status.h5ad",

    PROCESSED_DIR
    / "GBM_core_4datasets_fullgene_marker_based_annotation.h5ad",

    PROCESSED_DIR
    / "GBM_core_4datasets_fullgene_with_HVG8000_scVI_clusters.h5ad",
]

CANONICAL_CALLS_GZ = (
    CANONICAL_DIR
    / "copykat_microbatch_final_cell_calls_HVG8000_r0_6.tsv.gz"
)

CANONICAL_MANIFEST = (
    CANONICAL_DIR
    / "copykat_canonical_manifest.tsv"
)

CANONICAL_PREDICTION_BUNDLE = (
    CANONICAL_DIR
    / "copykat_microbatch_predictions_bundle_HVG8000_r0_6.tar.gz"
)

EXPECTED_BUNDLE_SHA256 = (
    "7b3f8c2b87847fcbcf3dff54a37dc077"
    "782e465da662cfcb9288914b9b9610b3"
)

# Standard canonical outputs consumed downstream.
OUT_ALL_ROWS_TSV = (
    METADATA_DIR
    / "copykat_microbatch_all_prediction_rows_HVG8000_r0_6.tsv"
)

OUT_TARGET_ONLY_TSV = (
    METADATA_DIR
    / "copykat_microbatch_target_only_predictions_HVG8000_r0_6.tsv"
)

OUT_FINAL_CELL_CALLS_TSV = (
    METADATA_DIR
    / "copykat_microbatch_final_cell_calls_HVG8000_r0_6.tsv"
)

OUT_OVERALL_SUMMARY_TSV = (
    METADATA_DIR
    / "copykat_microbatch_overall_summary_HVG8000_r0_6.tsv"
)

OUT_BATCH_QC_TSV = (
    METADATA_DIR
    / "copykat_microbatch_batch_qc_summary_HVG8000_r0_6.tsv"
)

OUT_CLUSTER_SUMMARY_TSV = (
    METADATA_DIR
    / "copykat_microbatch_cluster_summary_HVG8000_r0_6.tsv"
)

OUT_ANNOTATION_SUMMARY_TSV = (
    METADATA_DIR
    / "copykat_microbatch_annotation_summary_HVG8000_r0_6.tsv"
)

OUT_DATASET_SUMMARY_TSV = (
    METADATA_DIR
    / "copykat_microbatch_dataset_summary_HVG8000_r0_6.tsv"
)

OUT_CONDITION_SUMMARY_TSV = (
    METADATA_DIR
    / "copykat_microbatch_condition_summary_HVG8000_r0_6.tsv"
)

OUT_PATIENT_SUMMARY_TSV = (
    METADATA_DIR
    / "copykat_microbatch_patient_summary_HVG8000_r0_6.tsv"
)

OUT_CLUSTER_CONDITION_TSV = (
    METADATA_DIR
    / "copykat_microbatch_cluster_condition_summary_HVG8000_r0_6.tsv"
)

OUT_CLUSTER_INTERPRETATION_TSV = (
    METADATA_DIR
    / "copykat_microbatch_cluster_interpretation_HVG8000_r0_6.tsv"
)

OUT_OUTPUT_MANIFEST_TSV = (
    METADATA_DIR
    / "copykat_microbatch_integration_output_manifest_HVG8000_r0_6.tsv"
)

OUT_H5AD = (
    PROCESSED_DIR
    / "GBM_core_4datasets_fullgene_marker_based_annotation_bias_status_copykat_microbatch.h5ad"
)

CLUSTER_KEY_CANDIDATES = [
    "leiden_scvi_main",
    "leiden_scvi_HVG8000_r0_6",
    "leiden_scvi_0.6",
    "leiden",
]

ANNOTATION_COL_CANDIDATES = [
    "annotation_marker_based_HVG8000_r0_6",
    "annotation_marker_based",
    "cell_type_marker_based",
    "annotation",
]

DATASET_COL_CANDIDATES = [
    "core_dataset",
    "dataset_id",
    "dataset",
]

CONDITION_COL_CANDIDATES = [
    "condition",
    "disease",
    "diagnosis",
]

PATIENT_COL_CANDIDATES = [
    "patient_id",
    "patient",
    "Patient",
]


def first_existing(paths):
    for path in paths:
        path = Path(path)
        if path.exists():
            return path

    raise FileNotFoundError(
        "None of the candidate paths exists:\n"
        + "\n".join(map(str, paths))
    )


def choose_col(df, candidates, required=False, label="column"):
    for col in candidates:
        if col in df.columns:
            return col

    if required:
        raise KeyError(
            f"No {label} found. Tried: {candidates}. "
            f"Available columns: {list(df.columns)[:80]}"
        )

    return None


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(chunk_size)
            if not block:
                break
            digest.update(block)

    return digest.hexdigest()


def write_tsv_atomic(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    tmp = path.with_suffix(path.suffix + ".tmp")

    if tmp.exists():
        tmp.unlink()

    df.to_csv(
        tmp,
        sep="\t",
        index=False,
    )

    os.replace(tmp, path)

    print(
        "[SAVED]",
        path,
        "shape=",
        df.shape,
    )


def copy_file_with_progress(
    src,
    dst,
    chunk_size=64 * 1024 * 1024,
):
    src = Path(src)
    dst = Path(dst)

    total = src.stat().st_size
    copied = 0
    next_report = 5

    with src.open("rb") as fin, dst.open("wb") as fout:
        while True:
            block = fin.read(chunk_size)

            if not block:
                break

            fout.write(block)
            copied += len(block)

            pct = int(
                copied * 100 / max(1, total)
            )

            if pct >= next_report:
                print(
                    f"  {pct}% "
                    f"({copied / 1024**3:.2f} / "
                    f"{total / 1024**3:.2f} GiB)"
                )
                next_report += 5

        fout.flush()
        os.fsync(fout.fileno())

    shutil.copystat(src, dst)


def replace_h5ad_obs_group(h5ad_path, obs_df):
    with h5py.File(
        h5ad_path,
        "r+",
    ) as handle:

        if "obs" in handle:
            del handle["obs"]

        write_elem(
            handle,
            "obs",
            obs_df,
        )

        handle.flush()


print("PROJECT_DIR:", PROJECT_DIR)
print("Canonical directory:", CANONICAL_DIR)


PROJECT_DIR: G:\Repository\glioma-cnv-single-cell-atlas
Canonical directory: G:\Repository\glioma-cnv-single-cell-atlas\metadata\canonical


In [2]:
# =========================
# CANONICAL ARCHIVE INTEGRITY
# =========================
required_canonical = [
    CANONICAL_CALLS_GZ,
    CANONICAL_MANIFEST,
    CANONICAL_PREDICTION_BUNDLE,
]

missing = [
    path
    for path in required_canonical
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "Canonical CopyKAT archive is incomplete. Missing:\n"
        + "\n".join(map(str, missing))
        + "\nRun scripts/prepare_copykat_canonical_archive.py once "
        "during repository preparation, then commit metadata/canonical/."
    )

manifest = pd.read_csv(
    CANONICAL_MANIFEST,
    sep="\t",
)

print("Canonical manifest:")
display(manifest)

bundle_sha = sha256_file(
    CANONICAL_PREDICTION_BUNDLE
)

print(
    "Prediction bundle SHA256:",
    bundle_sha,
)

if bundle_sha != EXPECTED_BUNDLE_SHA256:
    raise RuntimeError(
        "Canonical historical prediction bundle checksum mismatch."
    )

call_row = manifest.loc[
    manifest["artifact"]
    == CANONICAL_CALLS_GZ.name
]

if len(call_row) != 1:
    raise RuntimeError(
        "Canonical manifest must contain exactly one final-cell-call artifact row."
    )

expected_calls_sha = str(
    call_row.iloc[0]["sha256"]
)

observed_calls_sha = sha256_file(
    CANONICAL_CALLS_GZ
)

print(
    "Canonical final-call SHA256:",
    observed_calls_sha,
)

if observed_calls_sha != expected_calls_sha:
    raise RuntimeError(
        "Canonical final-cell-call checksum mismatch."
    )

print(
    "PASS: canonical archive checksums validated."
)


Canonical manifest:


,artifact,sha256,size_bytes,role
0,copykat_microbatch_final_cell_calls_HVG8000_r0...,e1afd3c3378a02e31da2a28de32b2f27e4aba96236c5c1...,3535859,canonical paper-defining CopyKAT final cell calls
1,copykat_microbatch_predictions_bundle_HVG8000_...,7b3f8c2b87847fcbcf3dff54a37dc077782e465da662cf...,3387012,canonical archived 176-microbatch prediction p...


Prediction bundle SHA256: 7b3f8c2b87847fcbcf3dff54a37dc077782e465da662cfcb9288914b9b9610b3
Canonical final-call SHA256: e1afd3c3378a02e31da2a28de32b2f27e4aba96236c5c1c21f70cd50321e21f8
PASS: canonical archive checksums validated.


In [3]:
# =========================
# LOAD CANONICAL HISTORICAL CELL CALLS
# =========================
canonical = pd.read_csv(
    CANONICAL_CALLS_GZ,
    sep="\t",
    compression="gzip",
)

print(
    "Canonical final-call table:",
    canonical.shape,
)

required_cols = [
    "cell_id",
    "copykat_microbatch_pred",
]

missing_cols = [
    col
    for col in required_cols
    if col not in canonical.columns
]

if missing_cols:
    raise RuntimeError(
        "Canonical final-call table is missing required columns: "
        + ", ".join(missing_cols)
    )

canonical["cell_id"] = (
    canonical["cell_id"]
    .astype(str)
)

canonical[
    "copykat_microbatch_pred"
] = (
    canonical[
        "copykat_microbatch_pred"
    ]
    .astype(str)
    .str.strip()
)

if len(canonical) != 351427:
    raise RuntimeError(
        "Canonical final-call table must contain exactly 351,427 rows."
    )

if canonical["cell_id"].duplicated().any():
    raise RuntimeError(
        "Canonical final-call table contains duplicate raw cell IDs."
    )

EXPECTED_CALL_COUNTS = {
    "aneuploid": 202895,
    "diploid": 111493,
    "not.defined": 37037,
    "missing": 2,
}

observed_counts = (
    canonical[
        "copykat_microbatch_pred"
    ]
    .value_counts()
    .to_dict()
)

checkpoint = pd.DataFrame(
    [
        {
            "call": call,
            "expected": expected,
            "observed": int(
                observed_counts.get(
                    call,
                    0,
                )
            ),
            "exact_match": int(
                observed_counts.get(
                    call,
                    0,
                )
            )
            == expected,
        }
        for call, expected
        in EXPECTED_CALL_COUNTS.items()
    ]
)

display(checkpoint)

if not checkpoint[
    "exact_match"
].all():
    raise RuntimeError(
        "Canonical historical CopyKAT call-count checkpoint failed."
    )

print(
    "PASS: canonical historical CopyKAT hard checkpoints match exactly."
)


C:\Users\AmirSilco\AppData\Local\Temp\ipykernel_70444\943606911.py:4: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  canonical = pd.read_csv(


Canonical final-call table: (351427, 16)


,call,expected,observed,exact_match
0,aneuploid,202895,202895,True
1,diploid,111493,111493,True
2,not.defined,37037,37037,True
3,missing,2,2,True


PASS: canonical historical CopyKAT hard checkpoints match exactly.


In [4]:
# =========================
# LOAD SOURCE H5AD — OBS ONLY
# =========================
SOURCE_H5AD = first_existing(
    SOURCE_H5AD_CANDIDATES
)

print(
    "SOURCE_H5AD:",
    SOURCE_H5AD,
)

adata = sc.read_h5ad(
    SOURCE_H5AD,
    backed="r",
)

try:
    source_shape = adata.shape

    obs = (
        adata.obs
        .copy()
    )

    obs_names = (
        adata.obs_names
        .astype(str)
        .to_numpy()
        .copy()
    )

finally:
    adata.file.close()

if source_shape != (
    351427,
    15176,
):
    raise RuntimeError(
        f"Unexpected source H5AD shape: {source_shape}"
    )

obs.index = obs.index.astype(str)

if obs.index.has_duplicates:
    raise RuntimeError(
        "Source H5AD obs_names are duplicated."
    )

obs["cell_id"] = (
    obs.index
    .astype(str)
)

CLUSTER_KEY = choose_col(
    obs,
    CLUSTER_KEY_CANDIDATES,
    required=True,
    label="cluster key",
)

ANNOTATION_COL = choose_col(
    obs,
    ANNOTATION_COL_CANDIDATES,
)

DATASET_COL = choose_col(
    obs,
    DATASET_COL_CANDIDATES,
)

CONDITION_COL = choose_col(
    obs,
    CONDITION_COL_CANDIDATES,
)

PATIENT_COL = choose_col(
    obs,
    PATIENT_COL_CANDIDATES,
)

print("CLUSTER_KEY:", CLUSTER_KEY)
print("ANNOTATION_COL:", ANNOTATION_COL)
print("DATASET_COL:", DATASET_COL)
print("CONDITION_COL:", CONDITION_COL)
print("PATIENT_COL:", PATIENT_COL)


SOURCE_H5AD: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_fullgene_marker_based_annotation_bias_status.h5ad
CLUSTER_KEY: leiden_scvi_main
ANNOTATION_COL: annotation_marker_based_HVG8000_r0_6
DATASET_COL: core_dataset
CONDITION_COL: condition
PATIENT_COL: patient_id


In [5]:
# =========================
# STABLE BIOLOGICAL CELL IDENTITY
# =========================
def normalize_dataset(value):
    value = str(value)
    return (
        value
        .replace("DS5_GSE182109", "DS3_GSE182109")
        .replace("DS13_GSE278450", "DS4_GSE278450")
    )


def normalize_obs_name(value):
    value = str(value)
    return (
        value
        .replace("DS5_GSE182109", "DS3_GSE182109")
        .replace("DS13_GSE278450", "DS4_GSE278450")
    )


def normalize_sample(value):
    return re.sub(
        r"^GSM\d+_",
        "",
        str(value),
    )


def extract_nucleotide_barcode(value):
    candidates = re.findall(
        r"[ACGTN]{8,30}",
        str(value).upper(),
    )

    if not candidates:
        return None

    return max(candidates, key=len)


def canonical_cell_keys(
    df,
    label,
    force_unassigned_raw_ids=None,
):
    for required in [
        "cell_id",
        "core_dataset",
        "sample_id",
    ]:
        if required not in df.columns:
            raise RuntimeError(
                f"{label}: required column is missing: {required}"
            )

    force_unassigned_raw_ids = set(
        force_unassigned_raw_ids or []
    )

    normalized_raw_ids = (
        df["cell_id"]
        .astype(str)
        .map(normalize_obs_name)
    )

    datasets = (
        df["core_dataset"]
        .astype(str)
        .map(normalize_dataset)
    )

    keys = pd.Series(
        index=df.index,
        dtype="object",
    )

    # DS1 / DS3 / DS4 use normalized raw cell IDs.
    for dataset in [
        "DS1_GSE103224",
        "DS3_GSE182109",
        "DS4_GSE278450",
    ]:
        mask = datasets.eq(dataset)

        keys.loc[mask] = (
            dataset
            + "||"
            + normalized_raw_ids.loc[mask]
        )

    # DS2 requires normalized sample + nucleotide barcode because
    # historical/current raw cell IDs differ systematically.
    ds2_mask = datasets.eq(
        "DS2_GSE141383"
    )

    ds2_samples = (
        df.loc[
            ds2_mask,
            "sample_id",
        ]
        .astype(str)
        .map(normalize_sample)
    )

    ds2_barcodes = (
        df.loc[
            ds2_mask,
            "cell_id",
        ]
        .astype(str)
        .map(extract_nucleotide_barcode)
    )

    if ds2_barcodes.isna().any():
        raise RuntimeError(
            f"{label}: DS2 contains "
            f"{int(ds2_barcodes.isna().sum())} cells without "
            "an extractable nucleotide barcode."
        )

    keys.loc[
        ds2_mask
    ] = (
        "DS2_GSE141383||"
        + ds2_samples
        + "||"
        + ds2_barcodes
    )

    # Force the exact two historical CopyKAT-missing raw cell IDs to
    # use the same identity key on both canonical and source-H5AD sides.
    if force_unassigned_raw_ids:
        force_mask = (
            normalized_raw_ids
            .isin(force_unassigned_raw_ids)
        )

        observed_forced = set(
            normalized_raw_ids.loc[
                force_mask
            ]
        )

        if observed_forced != force_unassigned_raw_ids:
            missing_forced = sorted(
                force_unassigned_raw_ids
                - observed_forced
            )

            raise RuntimeError(
                f"{label}: not all historical missing-cell raw IDs "
                f"were found. Missing: {missing_forced}"
            )

        keys.loc[
            force_mask
        ] = (
            "UNASSIGNED_METADATA||"
            + normalized_raw_ids.loc[
                force_mask
            ]
        )

    # Any remaining unresolved rows use normalized raw ID.
    unresolved = keys.isna()

    if unresolved.any():
        keys.loc[
            unresolved
        ] = (
            "UNASSIGNED_METADATA||"
            + normalized_raw_ids.loc[
                unresolved
            ]
        )

    if keys.isna().any():
        raise RuntimeError(
            f"{label}: stable-key construction left "
            f"{int(keys.isna().sum())} unresolved rows."
        )

    if keys.duplicated().any():
        examples = (
            keys[
                keys.duplicated(
                    keep=False
                )
            ]
            .head(20)
            .tolist()
        )

        raise RuntimeError(
            f"{label}: stable biological cell keys are duplicated. "
            f"Examples: {examples}"
        )

    return keys


# Derive the two special cell IDs directly from the canonical archive.
canonical_missing_call_mask = (
    canonical[
        "copykat_microbatch_pred"
    ]
    .eq("missing")
)

if int(
    canonical_missing_call_mask.sum()
) != 2:
    raise RuntimeError(
        "Canonical archive must contain exactly two missing CopyKAT calls."
    )

historical_missing_raw_ids = set(
    canonical.loc[
        canonical_missing_call_mask,
        "cell_id",
    ]
    .astype(str)
    .map(normalize_obs_name)
)

if len(
    historical_missing_raw_ids
) != 2:
    raise RuntimeError(
        "The two canonical missing rows do not have two unique raw cell IDs."
    )

print(
    "Historical CopyKAT-missing raw cell IDs:"
)

for cell_id in sorted(
    historical_missing_raw_ids
):
    print(
        " ",
        cell_id,
    )


for required_col in [
    "core_dataset",
    "sample_id",
]:
    if required_col not in canonical.columns:
        raise RuntimeError(
            "Canonical final-call table lacks "
            f"{required_col}; rebuild the canonical archive from the "
            "historical Notebook-14 final-cell-call output."
        )


canonical[
    "canonical_cell_key"
] = canonical_cell_keys(
    canonical,
    "canonical_copykat",
    force_unassigned_raw_ids=historical_missing_raw_ids,
)

obs[
    "canonical_cell_key"
] = canonical_cell_keys(
    obs,
    "source_h5ad",
    force_unassigned_raw_ids=historical_missing_raw_ids,
)


canonical_keys = set(
    canonical[
        "canonical_cell_key"
    ]
)

source_keys = set(
    obs[
        "canonical_cell_key"
    ]
)

print(
    "Canonical unique stable IDs:",
    len(canonical_keys),
)

print(
    "Source H5AD unique stable IDs:",
    len(source_keys),
)

print(
    "Shared stable IDs:",
    len(canonical_keys & source_keys),
)

print(
    "Canonical-only stable IDs:",
    len(canonical_keys - source_keys),
)

print(
    "Source-only stable IDs:",
    len(source_keys - canonical_keys),
)

if canonical_keys != source_keys:
    print(
        "Canonical-only examples:",
        sorted(
            canonical_keys - source_keys
        )[:20],
    )

    print(
        "Source-only examples:",
        sorted(
            source_keys - canonical_keys
        )[:20],
    )

    raise RuntimeError(
        "Canonical CopyKAT calls and source H5AD do not represent "
        "the exact same biological cell set after the validated "
        "two-missing-cell identity override."
    )

print(
    "PASS: exact 351,427-cell biological identity match."
)


Historical CopyKAT-missing raw cell IDs:
  DS1_GSE103224:PJ017_cell001006
  DS1_GSE103224:PJ017_cell001173
Canonical unique stable IDs: 351427
Source H5AD unique stable IDs: 351427
Shared stable IDs: 351427
Canonical-only stable IDs: 0
Source-only stable IDs: 0
PASS: exact 351,427-cell biological identity match.


In [6]:
# =========================
# ALIGN CANONICAL CALLS TO CURRENT H5AD ORDER
# =========================
canonical_by_key = (
    canonical
    .set_index(
        "canonical_cell_key"
    )
    .reindex(
        obs[
            "canonical_cell_key"
        ]
    )
)

if canonical_by_key.index.has_duplicates:
    raise RuntimeError(
        "Aligned canonical CopyKAT table has duplicate stable IDs."
    )

if canonical_by_key[
    "copykat_microbatch_pred"
].isna().any():
    raise RuntimeError(
        "Aligned canonical CopyKAT table contains unexpected missing prediction values."
    )

final_calls = pd.DataFrame(
    {
        "cell_id": obs.index.astype(str),
        "copykat_microbatch_pred": (
            canonical_by_key[
                "copykat_microbatch_pred"
            ]
            .to_numpy()
        ),
    }
)

# Preserve historical per-cell CopyKAT provenance columns when available.
for col in [
    "copykat_microbatch_n_rows",
    "copykat_microbatch_batches",
]:
    if col in canonical_by_key.columns:
        final_calls[col] = (
            canonical_by_key[col]
            .to_numpy()
        )

if "copykat_microbatch_n_rows" not in final_calls.columns:
    final_calls[
        "copykat_microbatch_n_rows"
    ] = np.where(
        final_calls[
            "copykat_microbatch_pred"
        ]
        .eq("missing"),
        0,
        1,
    )

if "copykat_microbatch_batches" not in final_calls.columns:
    final_calls[
        "copykat_microbatch_batches"
    ] = ""

final_calls[
    "copykat_microbatch_defined"
] = (
    final_calls[
        "copykat_microbatch_pred"
    ]
    .isin(
        [
            "aneuploid",
            "diploid",
        ]
    )
)

final_calls[
    "copykat_microbatch_is_aneuploid"
] = (
    final_calls[
        "copykat_microbatch_pred"
    ]
    .eq("aneuploid")
)

final_calls[
    "copykat_microbatch_is_diploid"
] = (
    final_calls[
        "copykat_microbatch_pred"
    ]
    .eq("diploid")
)

final_calls[
    "copykat_microbatch_is_not_defined"
] = (
    final_calls[
        "copykat_microbatch_pred"
    ]
    .eq("not.defined")
)

final_calls[
    "copykat_microbatch_is_missing"
] = (
    final_calls[
        "copykat_microbatch_pred"
    ]
    .eq("missing")
)

# ------------------------------------------------------------
# Historical Notebook-14 summary metadata reconstruction
# ------------------------------------------------------------
# Historical summaries used the standardized analysis metadata carried
# by target predictions. The two cells without a target prediction were
# appended later and therefore had missing summary metadata.
#
# Reproduce that behavior exactly:
#   1) use the current canonical H5AD metadata for all cells;
#   2) set summary metadata to NaN only for the two CopyKAT-missing cells.
#
# This temporary final_calls table is used for summary generation only.
# The source H5AD biological metadata are not modified.
summary_metadata_cols = [
    CLUSTER_KEY,
    ANNOTATION_COL,
    DATASET_COL,
    CONDITION_COL,
    PATIENT_COL,
]

summary_metadata_audit_rows = []

for col in summary_metadata_cols:
    if (
        col is None
        or col not in obs.columns
    ):
        continue

    final_calls[col] = (
        obs[col]
        .to_numpy()
    )

    summary_metadata_audit_rows.append(
        {
            "column": col,
            "source": "current_canonical_h5ad",
            "n_missing_before_copykat_missing_override": int(
                pd.isna(
                    final_calls[col]
                ).sum()
            ),
        }
    )

missing_mask = (
    final_calls[
        "copykat_microbatch_is_missing"
    ]
)

if int(
    missing_mask.sum()
) != 2:
    raise RuntimeError(
        "Expected exactly two canonical CopyKAT-missing cells."
    )

for col in summary_metadata_cols:
    if (
        col is not None
        and col in final_calls.columns
    ):
        final_calls.loc[
            missing_mask,
            col,
        ] = np.nan

summary_metadata_audit = pd.DataFrame(
    summary_metadata_audit_rows
)

if len(summary_metadata_audit):
    summary_metadata_audit[
        "n_missing_after_copykat_missing_override"
    ] = [
        int(
            pd.isna(
                final_calls[row["column"]]
            ).sum()
        )
        for row
        in summary_metadata_audit_rows
    ]

print(
    "Historical summary metadata reconstruction:"
)

display(
    summary_metadata_audit
)

print(
    "Aligned canonical final calls:",
    final_calls.shape,
)

display(
    final_calls[
        "copykat_microbatch_pred"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis("call")
    .reset_index(name="n")
)

if len(final_calls) != 351427:
    raise RuntimeError(
        "Aligned canonical final calls must contain 351,427 cells."
    )

for call, expected in EXPECTED_CALL_COUNTS.items():
    observed = int(
        final_calls[
            "copykat_microbatch_pred"
        ]
        .eq(call)
        .sum()
    )

    if observed != expected:
        raise RuntimeError(
            f"Aligned call checkpoint failed for {call}: "
            f"{observed} != {expected}"
        )

print(
    "Two canonical CopyKAT-missing rows after summary-metadata override:"
)

display(
    final_calls.loc[
        missing_mask,
        [
            col
            for col in [
                "cell_id",
                CLUSTER_KEY,
                ANNOTATION_COL,
                DATASET_COL,
                CONDITION_COL,
                PATIENT_COL,
                "copykat_microbatch_pred",
            ]
            if (
                col is not None
                and col in final_calls.columns
            )
        ],
    ]
)


Historical summary metadata reconstruction:


,column,source,n_missing_before_copykat_missing_override,n_missing_after_copykat_missing_override
0,leiden_scvi_main,current_canonical_h5ad,0,2
1,annotation_marker_based_HVG8000_r0_6,current_canonical_h5ad,0,2
2,core_dataset,current_canonical_h5ad,0,2
3,condition,current_canonical_h5ad,0,2
4,patient_id,current_canonical_h5ad,0,2


Aligned canonical final calls: (351427, 14)


,call,n
0,aneuploid,202895
1,diploid,111493
2,not.defined,37037
3,missing,2


Two canonical CopyKAT-missing rows after summary-metadata override:


,cell_id,leiden_scvi_main,annotation_marker_based_HVG8000_r0_6,core_dataset,condition,patient_id,copykat_microbatch_pred
3828,DS1_GSE103224:PJ017_cell001006,NaN,NaN,NaN,NaN,NaN,missing
3990,DS1_GSE103224:PJ017_cell001173,NaN,NaN,NaN,NaN,NaN,missing


In [7]:
# =========================
# RECREATE HISTORICAL NOTEBOOK-14 SUMMARIES
# =========================
def summarize_by(df, group_cols):
    group_cols = [
        col
        for col in group_cols
        if (
            col is not None
            and col in df.columns
        )
    ]

    if not group_cols:
        return pd.DataFrame()

    tmp = df.copy()

    for col in group_cols:
        # Historical Notebook-14 behavior:
        # NaN metadata becomes the literal group label "nan".
        tmp[col] = (
            tmp[col]
            .astype(str)
        )

    out = (
        tmp
        .groupby(
            group_cols,
            observed=True,
        )
        .agg(
            n_cells=(
                "cell_id",
                "count",
            ),
            n_aneuploid=(
                "copykat_microbatch_is_aneuploid",
                "sum",
            ),
            n_diploid=(
                "copykat_microbatch_is_diploid",
                "sum",
            ),
            n_not_defined=(
                "copykat_microbatch_is_not_defined",
                "sum",
            ),
            n_missing=(
                "copykat_microbatch_is_missing",
                "sum",
            ),
        )
        .reset_index()
    )

    out["n_defined"] = (
        out["n_aneuploid"]
        + out["n_diploid"]
    )

    out[
        "aneuploid_fraction_all"
    ] = (
        out["n_aneuploid"]
        / out["n_cells"].replace(
            0,
            np.nan,
        )
    )

    out[
        "aneuploid_fraction_defined"
    ] = (
        out["n_aneuploid"]
        / out["n_defined"].replace(
            0,
            np.nan,
        )
    )

    out[
        "not_defined_fraction"
    ] = (
        out["n_not_defined"]
        / out["n_cells"].replace(
            0,
            np.nan,
        )
    )

    out[
        "missing_fraction"
    ] = (
        out["n_missing"]
        / out["n_cells"].replace(
            0,
            np.nan,
        )
    )

    return out.sort_values(
        [
            "aneuploid_fraction_defined",
            "n_cells",
        ],
        ascending=[
            False,
            False,
        ],
    )


cluster_summary = summarize_by(
    final_calls,
    [CLUSTER_KEY],
)

annotation_summary = summarize_by(
    final_calls,
    [ANNOTATION_COL],
)

dataset_summary = summarize_by(
    final_calls,
    [DATASET_COL],
)

condition_summary = summarize_by(
    final_calls,
    [CONDITION_COL],
)

patient_summary = summarize_by(
    final_calls,
    [PATIENT_COL],
)

cluster_condition_summary = summarize_by(
    final_calls,
    [
        CLUSTER_KEY,
        CONDITION_COL,
    ],
)

EXPECTED_SUMMARY_SHAPES = {
    "cluster_summary": (19, 11),
    "annotation_summary": (19, 11),
    "dataset_summary": (5, 11),
    "condition_summary": (5, 11),
    "patient_summary": (51, 11),
    "cluster_condition_summary": (73, 12),
}

observed_shapes = {
    "cluster_summary": cluster_summary.shape,
    "annotation_summary": annotation_summary.shape,
    "dataset_summary": dataset_summary.shape,
    "condition_summary": condition_summary.shape,
    "patient_summary": patient_summary.shape,
    "cluster_condition_summary": cluster_condition_summary.shape,
}

display(
    pd.DataFrame(
        [
            {
                "table": key,
                "expected_shape": str(
                    EXPECTED_SUMMARY_SHAPES[key]
                ),
                "observed_shape": str(
                    observed_shapes[key]
                ),
                "exact_match": (
                    observed_shapes[key]
                    == EXPECTED_SUMMARY_SHAPES[key]
                ),
            }
            for key in EXPECTED_SUMMARY_SHAPES
        ]
    )
)

shape_audit = pd.DataFrame(
    [
        {
            "table": key,
            "expected_rows": EXPECTED_SUMMARY_SHAPES[key][0],
            "expected_cols": EXPECTED_SUMMARY_SHAPES[key][1],
            "observed_rows": observed_shapes[key][0],
            "observed_cols": observed_shapes[key][1],
            "exact_match": (
                observed_shapes[key]
                == EXPECTED_SUMMARY_SHAPES[key]
            ),
        }
        for key in EXPECTED_SUMMARY_SHAPES
    ]
)

if not shape_audit[
    "exact_match"
].all():
    print(
        "Historical Notebook-14 summary-shape mismatches:"
    )

    display(
        shape_audit.loc[
            ~shape_audit[
                "exact_match"
            ]
        ]
    )

    raise RuntimeError(
        "Historical Notebook-14 summary-shape checkpoint failed."
    )

print(
    "PASS: all historical Notebook-14 summary shapes match exactly."
)


,table,expected_shape,observed_shape,exact_match
0,cluster_summary,"(19, 11)","(19, 11)",True
1,annotation_summary,"(19, 11)","(19, 11)",True
2,dataset_summary,"(5, 11)","(5, 11)",True
3,condition_summary,"(5, 11)","(5, 11)",True
4,patient_summary,"(51, 11)","(51, 11)",True
5,cluster_condition_summary,"(73, 12)","(73, 12)",True


PASS: all historical Notebook-14 summary shapes match exactly.


In [8]:
# =========================
# CLUSTER-LEVEL INTERPRETATION
# Historical thresholds preserved exactly.
# =========================
interpret = (
    cluster_summary
    .copy()
)

if (
    ANNOTATION_COL is not None
    and ANNOTATION_COL in final_calls.columns
):
    ann_map = (
        final_calls[
            [
                CLUSTER_KEY,
                ANNOTATION_COL,
            ]
        ]
        .dropna()
        .astype(str)
        .drop_duplicates()
        .groupby(
            CLUSTER_KEY
        )[
            ANNOTATION_COL
        ]
        .apply(
            lambda values:
                ";".join(
                    sorted(
                        set(values)
                    )
                )
        )
        .reset_index()
    )

    interpret = interpret.merge(
        ann_map,
        on=CLUSTER_KEY,
        how="left",
    )


def cluster_call(row):
    n_defined = row.get(
        "n_defined",
        0,
    )

    fraction = row.get(
        "aneuploid_fraction_defined",
        np.nan,
    )

    if (
        pd.isna(fraction)
        or n_defined < 30
    ):
        return "insufficient_defined_cells"

    if fraction >= 0.65:
        return "copykat_aneuploid_high"

    if fraction >= 0.40:
        return "copykat_aneuploid_mixed_moderate_high"

    if fraction >= 0.20:
        return "copykat_aneuploid_low_mixed"

    return "copykat_diploid_dominant"


interpret[
    "copykat_cluster_call"
] = interpret.apply(
    cluster_call,
    axis=1,
)


def biological_interpretation(row):
    annotation = (
        str(
            row.get(
                ANNOTATION_COL,
                "",
            )
        ).lower()
        if ANNOTATION_COL is not None
        else ""
    )

    call = row[
        "copykat_cluster_call"
    ]

    fraction = row.get(
        "aneuploid_fraction_defined",
        np.nan,
    )

    if (
        "tumor" in annotation
        and call
        in [
            "copykat_aneuploid_high",
            "copykat_aneuploid_mixed_moderate_high",
        ]
    ):
        return "marker_tumor_like_with_copykat_support"

    if (
        "tumor" in annotation
        and call
        == "copykat_aneuploid_low_mixed"
    ):
        return "marker_tumor_like_with_weak_or_mixed_copykat_support"

    if (
        any(
            token in annotation
            for token in [
                "endothelial",
                "oligodendrocyte",
                "myeloid",
                "t_nk",
                "b_plasma",
            ]
        )
        and fraction < 0.20
    ):
        return "non_malignant_reference_like_by_copykat"

    if call == "copykat_aneuploid_high":
        return "copykat_high_aneuploid_review_annotation"

    return "review_with_marker_and_composition_context"


interpret[
    "copykat_biological_interpretation"
] = interpret.apply(
    biological_interpretation,
    axis=1,
)

interpret = interpret.sort_values(
    [
        "aneuploid_fraction_defined",
        "n_cells",
    ],
    ascending=[
        False,
        False,
    ],
)

if interpret.shape != (
    19,
    14,
):
    raise RuntimeError(
        f"Cluster interpretation shape mismatch: {interpret.shape}"
    )

display(interpret)


,leiden_scvi_main,n_cells,n_aneuploid,n_diploid,n_not_defined,n_missing,n_defined,aneuploid_fraction_all,aneuploid_fraction_defined,not_defined_fraction,missing_fraction,annotation_marker_based_HVG8000_r0_6,copykat_cluster_call,copykat_biological_interpretation
0,4,32896,24754,4562,3580,0,29316,0.752493,0.844385,0.108828,0.0,putative_tumor_hypoxia_stress,copykat_aneuploid_high,marker_tumor_like_with_copykat_support
1,0,61646,45910,8947,6789,0,54857,0.744736,0.836903,0.110129,0.0,putative_tumor_OPC_NPC_like,copykat_aneuploid_high,marker_tumor_like_with_copykat_support
2,2,38634,29096,5913,3625,0,35009,0.753119,0.831101,0.093829,0.0,cycling_proliferating_cells,copykat_aneuploid_high,copykat_high_aneuploid_review_annotation
3,7,22524,14292,5140,3092,0,19432,0.634523,0.735488,0.137276,0.0,putative_tumor_AC_like_reactive,copykat_aneuploid_high,marker_tumor_like_with_copykat_support
4,10,6340,4445,1675,220,0,6120,0.701104,0.726307,0.034700,0.0,myeloid_FCN1_inflammatory_monocyte_like,copykat_aneuploid_high,copykat_high_aneuploid_review_annotation
5,11,6183,3646,1384,1153,0,5030,0.589681,0.724851,0.186479,0.0,putative_tumor_NPC_neuronal_like,copykat_aneuploid_high,marker_tumor_like_with_copykat_support
6,12,4803,2978,1446,379,0,4424,0.620029,0.673146,0.078909,0.0,MES_ECM_stromal_like_ambiguous,copykat_aneuploid_high,copykat_high_aneuploid_review_annotation
7,3,35354,21350,12033,1971,0,33383,0.603892,0.639547,0.055750,0.0,myeloid_inflammatory_MES_associated,copykat_aneuploid_mixed_moderate_high,review_with_marker_and_composition_context
8,1,59791,29402,24843,5546,0,54245,0.491746,0.542022,0.092756,0.0,myeloid_C1Q_TREM2_APOE_TAM,copykat_aneuploid_mixed_moderate_high,review_with_marker_and_composition_context
9,17,105,42,41,22,0,83,0.400000,0.506024,0.209524,0.0,erythroid_Hb_high_or_ambient_RBC,copykat_aneuploid_mixed_moderate_high,review_with_marker_and_composition_context


In [9]:
# =========================
# WRITE CANONICAL METADATA OUTPUTS
# =========================
overall = pd.DataFrame(
    [
        {
            "input_h5ad": str(SOURCE_H5AD),
            "canonical_call_archive": str(
                CANONICAL_CALLS_GZ
            ),
            "canonical_prediction_bundle": str(
                CANONICAL_PREDICTION_BUNDLE
            ),
            "canonical_prediction_bundle_sha256": (
                EXPECTED_BUNDLE_SHA256
            ),
            "n_adata_cells": 351427,
            "n_prediction_files": 176,
            "n_prediction_rows_total_including_anchors": 526425,
            "n_unique_cells_in_all_prediction_rows": 351425,
            "n_target_prediction_rows": 351425,
            "n_final_cells": 351427,
            "n_final_aneuploid": 202895,
            "n_final_diploid": 111493,
            "n_final_not_defined": 37037,
            "n_final_missing": 2,
            "aneuploid_fraction_all_cells": (
                202895 / 351427
            ),
            "aneuploid_fraction_defined_cells": (
                202895
                / (
                    202895
                    + 111493
                )
            ),
            "canonical_role": (
                "paper-defining archived historical CopyKAT result"
            ),
            "rerun_policy": (
                "later CopyKAT runtimes are sensitivity analyses and "
                "do not replace this canonical result automatically"
            ),
        }
    ]
)

# The archived final-call table is the canonical target-only per-cell table.
# Preserve its canonical aligned version as the standard final-cell-call output.
write_tsv_atomic(
    final_calls,
    OUT_FINAL_CELL_CALLS_TSV,
)

# Compatibility outputs:
# all_prediction_rows and target_only_predictions historically existed.
# The full repeated-anchor table is preserved inside the canonical prediction
# bundle rather than expanded into a >200 MB working TSV in the main pipeline.
target_compat = (
    final_calls.loc[
        ~final_calls[
            "copykat_microbatch_is_missing"
        ]
    ]
    .copy()
)

write_tsv_atomic(
    target_compat,
    OUT_TARGET_ONLY_TSV,
)

all_rows_pointer = pd.DataFrame(
    [
        {
            "storage_mode": "archived_bundle",
            "canonical_prediction_bundle": str(
                CANONICAL_PREDICTION_BUNDLE
            ),
            "sha256": EXPECTED_BUNDLE_SHA256,
            "n_prediction_files": 176,
            "n_prediction_rows_total_including_anchors": 526425,
            "note": (
                "Repeated-anchor prediction rows are retained in the "
                "checksum-validated canonical tar.gz bundle and are not "
                "expanded into a >200 MB main-pipeline TSV."
            ),
        }
    ]
)

write_tsv_atomic(
    all_rows_pointer,
    OUT_ALL_ROWS_TSV,
)

batch_qc = pd.DataFrame(
    [
        {
            "canonical_source": "archived_bundle",
            "n_prediction_files": 176,
            "n_prediction_rows": 526425,
            "n_unique_prediction_cells": 351425,
            "bundle_sha256": EXPECTED_BUNDLE_SHA256,
            "status": "canonical_archive_validated",
        }
    ]
)

write_tsv_atomic(
    batch_qc,
    OUT_BATCH_QC_TSV,
)

write_tsv_atomic(
    overall,
    OUT_OVERALL_SUMMARY_TSV,
)

write_tsv_atomic(
    cluster_summary,
    OUT_CLUSTER_SUMMARY_TSV,
)

write_tsv_atomic(
    annotation_summary,
    OUT_ANNOTATION_SUMMARY_TSV,
)

write_tsv_atomic(
    dataset_summary,
    OUT_DATASET_SUMMARY_TSV,
)

write_tsv_atomic(
    condition_summary,
    OUT_CONDITION_SUMMARY_TSV,
)

write_tsv_atomic(
    patient_summary,
    OUT_PATIENT_SUMMARY_TSV,
)

write_tsv_atomic(
    cluster_condition_summary,
    OUT_CLUSTER_CONDITION_TSV,
)

write_tsv_atomic(
    interpret,
    OUT_CLUSTER_INTERPRETATION_TSV,
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\copykat_microbatch_final_cell_calls_HVG8000_r0_6.tsv shape= (351427, 14)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\copykat_microbatch_target_only_predictions_HVG8000_r0_6.tsv shape= (351425, 14)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\copykat_microbatch_all_prediction_rows_HVG8000_r0_6.tsv shape= (1, 6)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\copykat_microbatch_batch_qc_summary_HVG8000_r0_6.tsv shape= (1, 6)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\copykat_microbatch_overall_summary_HVG8000_r0_6.tsv shape= (1, 18)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\copykat_microbatch_cluster_summary_HVG8000_r0_6.tsv shape= (19, 11)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\copykat_microbatch_annotation_summary_HVG8000_r0_6.tsv shape= (19, 11)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\copykat_microbatch_datas

In [10]:
# =========================
# BUILD CANONICAL H5AD — MEMORY SAFE
# =========================
if OUT_H5AD.exists():
    existing = sc.read_h5ad(
        OUT_H5AD,
        backed="r",
    )

    try:
        existing_counts = (
            existing.obs[
                "copykat_microbatch_pred"
            ]
            .astype(str)
            .value_counts()
            .to_dict()
            if "copykat_microbatch_pred"
            in existing.obs.columns
            else {}
        )
    finally:
        existing.file.close()

    canonical_existing = all(
        int(
            existing_counts.get(
                call,
                0,
            )
        )
        == expected
        for call, expected
        in EXPECTED_CALL_COUNTS.items()
    )

    if not canonical_existing:
        raise RuntimeError(
            "A non-canonical Notebook-14 H5AD already exists at the standard "
            "output path. Run code/validation/copykat_sensitivity/"
            "archive_current_copykat_sensitivity.py first so that the "
            "sensitivity rerun is preserved rather than overwritten."
        )

output_obs = (
    obs
    .drop(
        columns=[
            "cell_id",
            "canonical_cell_key",
        ],
        errors="ignore",
    )
    .copy()
)

fc = (
    final_calls
    .set_index(
        "cell_id"
    )
    .reindex(
        output_obs.index.astype(str)
    )
)

for col in [
    "copykat_microbatch_pred",
    "copykat_microbatch_n_rows",
    "copykat_microbatch_batches",
    "copykat_microbatch_defined",
    "copykat_microbatch_is_aneuploid",
    "copykat_microbatch_is_diploid",
    "copykat_microbatch_is_not_defined",
    "copykat_microbatch_is_missing",
]:
    if col in fc.columns:
        output_obs[col] = (
            fc[col]
            .to_numpy()
        )

output_obs[
    "copykat_canonical_source"
] = "archived_historical_paper_run"

for col in [
    "copykat_microbatch_pred",
    "copykat_microbatch_batches",
    "copykat_canonical_source",
]:
    if col in output_obs.columns:
        output_obs[col] = (
            output_obs[col]
            .astype("category")
        )

if not np.array_equal(
    output_obs.index.astype(str),
    obs_names,
):
    raise RuntimeError(
        "Output obs order no longer matches source H5AD obs order."
    )

tmp_h5ad = OUT_H5AD.with_name(
    OUT_H5AD.stem
    + ".building.h5ad"
)

if tmp_h5ad.exists():
    tmp_h5ad.unlink()

required_bytes = int(
    SOURCE_H5AD.stat().st_size
    * 1.05
)

free_bytes = shutil.disk_usage(
    OUT_H5AD.parent
).free

print(
    "Source H5AD GiB:",
    round(
        SOURCE_H5AD.stat().st_size
        / 1024**3,
        3,
    ),
)

print(
    "Free space GiB:",
    round(
        free_bytes
        / 1024**3,
        3,
    ),
)

if free_bytes < required_bytes:
    raise RuntimeError(
        "Insufficient free space for safe atomic H5AD creation."
    )

print(
    "[COPY SOURCE H5AD]"
)

copy_file_with_progress(
    SOURCE_H5AD,
    tmp_h5ad,
)

print(
    "[REPLACE /obs ONLY]"
)

replace_h5ad_obs_group(
    tmp_h5ad,
    output_obs,
)

print(
    "[BACKED VALIDATION]"
)

check = sc.read_h5ad(
    tmp_h5ad,
    backed="r",
)

try:
    if check.shape != (
        351427,
        15176,
    ):
        raise RuntimeError(
            f"Output H5AD shape mismatch: {check.shape}"
        )

    if not np.array_equal(
        check.obs_names.astype(str),
        output_obs.index.astype(str),
    ):
        raise RuntimeError(
            "Output H5AD obs order mismatch."
        )

    check_counts = (
        check.obs[
            "copykat_microbatch_pred"
        ]
        .astype(str)
        .value_counts()
        .to_dict()
    )

    for call, expected in EXPECTED_CALL_COUNTS.items():
        observed = int(
            check_counts.get(
                call,
                0,
            )
        )

        if observed != expected:
            raise RuntimeError(
                f"Backed H5AD checkpoint failed for {call}: "
                f"{observed} != {expected}"
            )

finally:
    check.file.close()

os.replace(
    tmp_h5ad,
    OUT_H5AD,
)

print(
    "[SAVED]",
    OUT_H5AD,
)


Source H5AD GiB: 12.301
Free space GiB: 435.681
[COPY SOURCE H5AD]
  5% (0.62 / 12.30 GiB)
  10% (1.25 / 12.30 GiB)
  15% (1.88 / 12.30 GiB)
  20% (2.50 / 12.30 GiB)
  25% (3.12 / 12.30 GiB)
  30% (3.75 / 12.30 GiB)
  35% (4.31 / 12.30 GiB)
  40% (4.94 / 12.30 GiB)
  45% (5.56 / 12.30 GiB)
  50% (6.19 / 12.30 GiB)
  55% (6.81 / 12.30 GiB)
  60% (7.44 / 12.30 GiB)
  65% (8.00 / 12.30 GiB)
  70% (8.62 / 12.30 GiB)
  75% (9.25 / 12.30 GiB)
  80% (9.88 / 12.30 GiB)
  85% (10.50 / 12.30 GiB)
  90% (11.12 / 12.30 GiB)
  95% (11.69 / 12.30 GiB)
  100% (12.30 / 12.30 GiB)
[REPLACE /obs ONLY]
[BACKED VALIDATION]
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_fullgene_marker_based_annotation_bias_status_copykat_microbatch.h5ad


In [11]:
# =========================
# OUTPUT MANIFEST + FINAL CHECKPOINTS
# =========================
manifest_rows = []

for label, path in [
    (
        "canonical_archive_manifest",
        CANONICAL_MANIFEST,
    ),
    (
        "canonical_prediction_bundle",
        CANONICAL_PREDICTION_BUNDLE,
    ),
    (
        "canonical_final_calls_gzip",
        CANONICAL_CALLS_GZ,
    ),
    (
        "all_prediction_rows_archive_pointer",
        OUT_ALL_ROWS_TSV,
    ),
    (
        "target_only_predictions",
        OUT_TARGET_ONLY_TSV,
    ),
    (
        "final_cell_calls",
        OUT_FINAL_CELL_CALLS_TSV,
    ),
    (
        "overall_summary",
        OUT_OVERALL_SUMMARY_TSV,
    ),
    (
        "batch_qc_summary",
        OUT_BATCH_QC_TSV,
    ),
    (
        "cluster_summary",
        OUT_CLUSTER_SUMMARY_TSV,
    ),
    (
        "annotation_summary",
        OUT_ANNOTATION_SUMMARY_TSV,
    ),
    (
        "dataset_summary",
        OUT_DATASET_SUMMARY_TSV,
    ),
    (
        "condition_summary",
        OUT_CONDITION_SUMMARY_TSV,
    ),
    (
        "patient_summary",
        OUT_PATIENT_SUMMARY_TSV,
    ),
    (
        "cluster_condition_summary",
        OUT_CLUSTER_CONDITION_TSV,
    ),
    (
        "cluster_interpretation",
        OUT_CLUSTER_INTERPRETATION_TSV,
    ),
    (
        "canonical_h5ad",
        OUT_H5AD,
    ),
]:
    path = Path(path)

    manifest_rows.append(
        {
            "output_label": label,
            "path": str(path),
            "exists": path.exists(),
            "size_mb": (
                round(
                    path.stat().st_size
                    / 1024**2,
                    3,
                )
                if path.exists()
                else np.nan
            ),
        }
    )

out_manifest = pd.DataFrame(
    manifest_rows
)

write_tsv_atomic(
    out_manifest,
    OUT_OUTPUT_MANIFEST_TSV,
)

display(out_manifest)

if not out_manifest[
    "exists"
].all():
    raise RuntimeError(
        "One or more expected canonical Notebook-14 outputs are missing."
    )

print("\nFINAL CHECKPOINTS")
print("Canonical prediction files: 176")
print("Historical prediction rows including repeated anchors: 526425")
print("Historical unique prediction cells: 351425")
print("Canonical final cells: 351427")
print("Canonical aneuploid: 202895")
print("Canonical diploid: 111493")
print("Canonical not.defined: 37037")
print("Canonical missing: 2")
print("Cluster summary:", cluster_summary.shape)
print("Annotation summary:", annotation_summary.shape)
print("Dataset summary:", dataset_summary.shape)
print("Condition summary:", condition_summary.shape)
print("Patient summary:", patient_summary.shape)
print("Cluster-condition summary:", cluster_condition_summary.shape)
print("Cluster interpretation:", interpret.shape)
print("Canonical H5AD:", OUT_H5AD)
print(
    "\n[DONE] Canonical archived CopyKAT integration completed. "
    "Notebook 15 may now consume the standard CopyKAT H5AD."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\copykat_microbatch_integration_output_manifest_HVG8000_r0_6.tsv shape= (16, 4)


,output_label,path,exists,size_mb
0,canonical_archive_manifest,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000
1,canonical_prediction_bundle,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,3.230
2,canonical_final_calls_gzip,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,3.372
3,all_prediction_rows_archive_pointer,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000
4,target_only_predictions,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,58.661
5,final_cell_calls,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,58.662
6,overall_summary,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.001
7,batch_qc_summary,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000
8,cluster_summary,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.002
9,annotation_summary,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.002



FINAL CHECKPOINTS
Canonical prediction files: 176
Historical prediction rows including repeated anchors: 526425
Historical unique prediction cells: 351425
Canonical final cells: 351427
Canonical aneuploid: 202895
Canonical diploid: 111493
Canonical not.defined: 37037
Canonical missing: 2
Cluster summary: (19, 11)
Annotation summary: (19, 11)
Dataset summary: (5, 11)
Condition summary: (5, 11)
Patient summary: (51, 11)
Cluster-condition summary: (73, 12)
Cluster interpretation: (19, 14)
Canonical H5AD: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_fullgene_marker_based_annotation_bias_status_copykat_microbatch.h5ad

[DONE] Canonical archived CopyKAT integration completed. Notebook 15 may now consume the standard CopyKAT H5AD.
